# Bluesky: the open firehose

The most open social corpus of the post-API era: the Jetstream firehose needs **no API
key**, and a one-hour pull collects hundreds of thousands of public posts. This starter
runs offline on a recorded slice so the pipeline is readable anywhere.

In [ ]:
import os, json
import pandas as pd
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
TRY_LIVE = not SMOKE and os.environ.get("BLUESKY_LIVE") == "1"   # opt in explicitly

RECORDED = [
 {"text": "the new album is a masterpiece, no notes", "langs": ["en"], "createdAt": "2026-05-01T12:00:01Z"},
 {"text": "museum day. the rothko room got me again", "langs": ["en"], "createdAt": "2026-05-01T12:00:03Z"},
 {"text": "hot take: the sequel is better than the original", "langs": ["en"], "createdAt": "2026-05-01T12:00:04Z"},
]

In [ ]:
def collect(seconds=30, max_posts=2000):
    """Collect public posts from the Jetstream firehose; offline, the recorded slice."""
    if not TRY_LIVE:
        print("offline: using the recorded slice (set BLUESKY_LIVE=1 for a real pull)")
        return pd.DataFrame(RECORDED)
    import time, websockets.sync.client as ws   # needs: pip install websockets
    url = "wss://jetstream2.us-east.bsky.network/subscribe?wantedCollections=app.bsky.feed.post"
    rows, t0 = [], time.time()
    with ws.connect(url) as sock:
        while time.time() - t0 < seconds and len(rows) < max_posts:
            msg = json.loads(sock.recv())
            rec = msg.get("commit", {}).get("record", {})
            if rec.get("$type") == "app.bsky.feed.post":
                rows.append({"text": rec.get("text", ""), "langs": rec.get("langs"),
                             "createdAt": rec.get("createdAt")})
    return pd.DataFrame(rows)

posts = collect()
print(len(posts), "posts")
posts.head()

**What a slice can carry:** a live hour is a genuine sample of *right now* — good for
language questions, event reactions, community vocabularies. It is also only that hour:
name the window in your Data Biography, because time-of-day and day-of-week shape who is
posting.